# EL-GHALI MOHAMED

### ARBRE DE DECISION FROM SCRATCH (ID3)

In [13]:
import pandas as pd
import math
from pprint import pprint

#  Création du dataset
data = {
    'Perspectives': ['Ensoleille', 'Ensoleille', 'Couvert', 'Pluie', 'Pluie', 'Pluie', 'Couvert',
                     'Ensoleille', 'Ensoleille', 'Pluie', 'Ensoleille', 'Couvert', 'Couvert', 'Pluie'],
    'Temperature': ['Chaude', 'Chaude', 'Chaude', 'Douce', 'Fraiche', 'Fraiche', 'Fraiche',
                    'Douce', 'Fraiche', 'Douce', 'Douce', 'Douce', 'Chaude', 'Douce'],
    'Humidite': ['Haute', 'Haute', 'Haute', 'Haute', 'Normale', 'Normale', 'Normale',
                 'Haute', 'Normale', 'Normale', 'Normale', 'Haute', 'Normale', 'Haute'],
    'Vent': ['Faible', 'Fort', 'Faible', 'Faible', 'Faible', 'Fort', 'Fort',
             'Faible', 'Faible', 'Faible', 'Fort', 'Fort', 'Faible', 'Fort'],
    'Jouer': ['Non', 'Non', 'Oui', 'Oui', 'Oui', 'Non', 'Oui',
              'Non', 'Oui', 'Oui', 'Oui', 'Oui', 'Oui', 'Non']
}

#  Conversion en DataFrame Pandas
df = pd.DataFrame(data)
#  Préparation des données pour votre modèle ID3
# On sépare les caractéristiques  de la variable cible
X = df.drop('Jouer', axis=1)  # Les caractéristiques : Perspectives, Température, Humidité, Vent
y = df['Jouer']               #  Jouer (Oui/Non)

### LES FONCTION MATHEMATIQUES

1. **L'Entropie ($H$)** : Elle mesure l'impureté ou le désordre dans le jeu de données. Plus les données sont mélangées (ex: 50% Oui, 50% Non), plus l'entropie est proche de 1.

$$H(S) = -\sum_{i=1}^{c} p_i \log_2(p_i)$$


2. **Le Gain d'Information ($Gain$)** : Il mesure la réduction de l'entropie après avoir divisé les données selon un attribut spécifique. L'ID3 choisit toujours l'attribut qui maximise ce gain.

$$Gain(S, A) = H(S) - \sum_{v \in Valeurs(A)} \frac{|S_v|}{|S|} H(S_v)$$







In [8]:
def calcul_entropie(colonne_cible):
    """Calcule l'entropie d'un ensemble de données."""
    occurrences = colonne_cible.value_counts().tolist()
    total = sum(occurrences)
    entropie = 0
    for occ in occurrences:
        probabilite = occ / total
        entropie -= probabilite * math.log2(probabilite)
    return entropie


In [9]:
def gain_information(donnees, attribut_split, nom_cible):
    """Calcule le gain d'information d'un attribut."""
    # Entropie du noeud parent
    entropie_totale = calcul_entropie(donnees[nom_cible])

    # Calcul de l'entropie pondérée des enfants
    valeurs = donnees[attribut_split].unique()
    entropie_ponderee = 0
    total_lignes = len(donnees)

    for val in valeurs:
        sous_ensemble = donnees[donnees[attribut_split] == val]
        probabilite_sous_ensemble = len(sous_ensemble) / total_lignes
        entropie_ponderee += probabilite_sous_ensemble * calcul_entropie(sous_ensemble[nom_cible])

    # Le gain est la différence entre l'entropie totale et l'entropie pondérée
    return entropie_totale - entropie_ponderee

### L'ALGORITHME DE CONSTRUCTION DE L'ARBRE

In [10]:
def construire_arbre_id3(donnees, dataset_original, colonnes_features, nom_cible, classe_parent=None):
    """Fonction récursive pour construire l'arbre de décision."""

    # Cas de base 1 : Si toutes les cibles ont la même valeur, on retourne cette valeur (feuille)
    if len(donnees[nom_cible].unique()) <= 1:
        return donnees[nom_cible].iloc[0]

    # Cas de base 2 : Si les données sont vides, on retourne la valeur majoritaire du parent
    elif len(donnees) == 0:
        return dataset_original[nom_cible].value_counts().idxmax()

    # Cas de base 3 : S'il n'y a plus de features à tester, on retourne la classe majoritaire actuelle
    elif len(colonnes_features) == 0:
        return donnees[nom_cible].value_counts().idxmax()

    # Construction récursive de l'arbre
    else:
        # Classe majoritaire du noeud actuel (utilisée si les futurs enfants sont vides)
        classe_parent = donnees[nom_cible].value_counts().idxmax()

        # Trouver le meilleur attribut (celui avec le plus grand gain d'information)
        gains = [gain_information(donnees, feature, nom_cible) for feature in colonnes_features]
        index_meilleur_feature = gains.index(max(gains))
        meilleur_feature = colonnes_features[index_meilleur_feature]

        # Créer la structure du noeud (dictionnaire)
        arbre = {meilleur_feature: {}}

        # Retirer l'attribut qui vient d'être utilisé pour les prochaines itérations
        features_restantes = [f for f in colonnes_features if f != meilleur_feature]

        # Créer les branches pour chaque valeur possible du meilleur attribut
        for valeur in donnees[meilleur_feature].unique():
            sous_donnees = donnees[donnees[meilleur_feature] == valeur]

            # Appel récursif
            sous_arbre = construire_arbre_id3(sous_donnees, dataset_original, features_restantes, nom_cible, classe_parent)

            # Attacher le sous-arbre à la branche correspondante
            arbre[meilleur_feature][valeur] = sous_arbre

        return arbre

### FONCTION PREDICTION

In [11]:
def predire(requete, arbre):
    """Navigue dans l'arbre pour prédire la classe d'une nouvelle instance."""
    for cle in list(requete.keys()):
        if cle in list(arbre.keys()):
            try:
                resultat = arbre[cle][requete[cle]]
            except KeyError:
                return "Inconnu (valeur non vue lors de l'entraînement)"

            # Si le résultat est encore un dictionnaire, on descend plus bas dans l'arbre
            if isinstance(resultat, dict):
                return predire(requete, resultat)
            else:
                return resultat # On a atteint une feuille

### EXÉCUTION ET TEST

In [12]:
features = ['Perspectives', 'Temperature', 'Humidite', 'Vent']
cible = 'Jouer'

# Entraînement du modèle
arbre_decision = construire_arbre_id3(df, df, features, cible)

print(" Arbre de Décision ")
pprint(arbre_decision)
print("-" * 50)

# Test de prédiction avec de nouvelles données
nouvelle_journee_1 = {'Perspectives': 'Ensoleille', 'Temperature': 'Fraiche', 'Humidite': 'Normale', 'Vent': 'Faible'}
nouvelle_journee_2 = {'Perspectives': 'Pluie', 'Temperature': 'Douce', 'Humidite': 'Haute', 'Vent': 'Fort'}

print(f"Prédiction pour la journée 1 {nouvelle_journee_1} :")
print("=> Jouer football ? :", predire(nouvelle_journee_1, arbre_decision))
print("\n")

print(f"Prédiction pour la journée 2 {nouvelle_journee_2} :")
print("=> Jouer football ? :", predire(nouvelle_journee_2, arbre_decision))

 Arbre de Décision 
{'Perspectives': {'Couvert': 'Oui',
                  'Ensoleille': {'Humidite': {'Haute': 'Non',
                                              'Normale': 'Oui'}},
                  'Pluie': {'Vent': {'Faible': 'Oui', 'Fort': 'Non'}}}}
--------------------------------------------------
Prédiction pour la journée 1 {'Perspectives': 'Ensoleille', 'Temperature': 'Fraiche', 'Humidite': 'Normale', 'Vent': 'Faible'} :
=> Jouer football ? : Oui


Prédiction pour la journée 2 {'Perspectives': 'Pluie', 'Temperature': 'Douce', 'Humidite': 'Haute', 'Vent': 'Fort'} :
=> Jouer football ? : Non
